In [0]:
from pyspark.sql import functions as F

# ============================================================
# Step 1: Configuration
# ============================================================

source_path = "s3://ujjivanpoc/source/bank_transactions/"

target_path = "s3://ujjivanpoc/Bronze/bank_transaction_fraud_detection"

target_table = "ujjivan_2.bronze.bank_transaction_fraud_detection"

checkpoint_path = "s3://ujjivanpoc/Bronze/_checkpoints/bank_transaction_fraud_detection"

schema_location = "s3://ujjivanpoc/Bronze/_schemas/bank_transaction_fraud_detection"


# IMPORTANT:
# False = Normal incremental load
# True  = Reset Auto Loader and process all files again
#
# Keep this FALSE for normal production/incremental runs.
RESET_CHECKPOINT = False


# ============================================================
# Step 2: Create Bronze table if it does not already exist
# ============================================================

if not spark.catalog.tableExists(target_table):

    print(f"Creating Bronze table: {target_table}")

    schema_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(source_path)
        .limit(0)
        .withColumn(
            "_source_file",
            F.lit(None).cast("string")
        )
        .withColumn(
            "_ingested_at",
            F.lit(None).cast("timestamp")
        )
        .withColumn(
            "_rescued_data",
            F.lit(None).cast("string")
        )
    )

    (
        schema_df.write
        .format("delta")
        .mode("overwrite")
        .option("path", target_path)
        .saveAsTable(target_table)
    )

    print(f"✅ Created table: {target_table}")
    print(f"📍 Location: {target_path}")

else:

    print(f"ℹ️ Table already exists: {target_table}")
    print("Proceeding with incremental load...")


# ============================================================
# Step 3: Reset Auto Loader checkpoint if requested
# ============================================================

if RESET_CHECKPOINT:

    print("\n⚠️ RESET_CHECKPOINT = True")
    print("Clearing Auto Loader checkpoint and schema location...")

    try:
        dbutils.fs.rm(checkpoint_path, recurse=True)
        print("✅ Checkpoint removed")
    except Exception as e:
        print(f"ℹ️ Checkpoint removal message: {e}")

    try:
        dbutils.fs.rm(schema_location, recurse=True)
        print("✅ Schema location removed")
    except Exception as e:
        print(f"ℹ️ Schema location removal message: {e}")

    print(
        "\n⚠️ Auto Loader will rediscover all files in the source path."
    )

else:

    print(
        "\n✅ Checkpoint/schema preserved."
    )
    print(
        "Only new/unseen files will be processed."
    )


# ============================================================
# Step 4: Row count BEFORE load
# ============================================================

count_before = (
    spark.table(target_table)
    .count()
)

print("\n" + "=" * 70)
print("BEFORE LOAD")
print("=" * 70)

print(f"📊 Row count BEFORE load: {count_before}")


# ============================================================
# Step 5: Create Auto Loader streaming DataFrame
# ============================================================

print("\n" + "=" * 70)
print("STARTING AUTO LOADER")
print("=" * 70)

df_stream = (
    spark.readStream
    .format("cloudFiles")

    # Source file format
    .option(
        "cloudFiles.format",
        "csv"
    )

    # CSV header
    .option(
        "header",
        "true"
    )

    # Automatically infer data types
    .option(
        "cloudFiles.inferColumnTypes",
        "true"
    )

    # Auto Loader schema location
    .option(
        "cloudFiles.schemaLocation",
        schema_location
    )

    # Store unexpected/malformed data
    .option(
        "cloudFiles.rescuedDataColumn",
        "_rescued_data"
    )

    # Load source files
    .load(source_path)

    # Add source file name/path
    .withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )

    # Add ingestion timestamp
    .withColumn(
        "_ingested_at",
        F.current_timestamp()
    )
)


# ============================================================
# Step 6: Write Auto Loader stream into Bronze Delta table
# ============================================================

print("Writing data into Bronze Delta table...")

query = (
    df_stream.writeStream

    # Delta format
    .format("delta")

    # Auto Loader checkpoint
    .option(
        "checkpointLocation",
        checkpoint_path
    )

    # Allow schema evolution
    .option(
        "mergeSchema",
        "true"
    )

    # Process currently available files and stop
    .trigger(
        availableNow=True
    )

    # Target Delta table
    .toTable(
        target_table
    )
)


# ============================================================
# Step 7: Wait until streaming load completes
# ============================================================

print("Waiting for Auto Loader to complete...")

query.awaitTermination()

print("✅ Auto Loader processing completed")


# ============================================================
# Step 8: Row count AFTER load
# ============================================================

count_after = (
    spark.table(target_table)
    .count()
)

rows_inserted = (
    count_after - count_before
)


print("\n" + "=" * 70)
print("AFTER LOAD")
print("=" * 70)

print(f"📊 Row count BEFORE load : {count_before}")
print(f"📊 Row count AFTER load  : {count_after}")
print(f"📈 Rows inserted         : {rows_inserted}")


# ============================================================
# Step 9: Micro-batch details
# ============================================================

print("\n" + "=" * 70)
print("MICRO-BATCH DETAILS")
print("=" * 70)

progress_list = query.recentProgress


if not progress_list:

    print(
        "⚠️ No progress recorded."
    )

    print(
        "This usually means no new files were available."
    )

else:

    print(
        f"🔍 Processed {len(progress_list)} micro-batch(es)"
    )

    total_input_rows = 0

    for i, p in enumerate(
        progress_list,
        start=1
    ):

        # ----------------------------------------------------
        # Safely get input row count
        # ----------------------------------------------------

        num_input_rows = p.get("numInputRows")

        # FIX:
        # numInputRows can sometimes be None.
        if num_input_rows is None:
            num_input_rows = 0

        # Convert to integer
        try:
            num_input_rows = int(num_input_rows)
        except Exception:
            num_input_rows = 0


        # ----------------------------------------------------
        # Add to total
        # ----------------------------------------------------

        total_input_rows += num_input_rows


        # ----------------------------------------------------
        # Safely get source information
        # ----------------------------------------------------

        sources = p.get("sources")

        if not sources:
            sources = [{}]

        source = sources[0]


        # ----------------------------------------------------
        # Batch duration
        # ----------------------------------------------------

        batch_duration = p.get(
            "batchDuration"
        )

        if batch_duration is None:
            batch_duration = 0


        # ----------------------------------------------------
        # Latest offset
        # ----------------------------------------------------

        latest_offset = source.get(
            "latestOffset"
        )

        if latest_offset is None:
            latest_offset = "N/A"


        # ----------------------------------------------------
        # Source description
        # ----------------------------------------------------

        description = source.get(
            "description"
        )

        if description is None:
            description = "N/A"

        description = str(description)


        # ----------------------------------------------------
        # Print batch information
        # ----------------------------------------------------

        print(
            f"\nBatch {i}"
        )

        print(
            f"  Input Rows     : {num_input_rows}"
        )

        print(
            f"  Batch Duration : {batch_duration} ms"
        )

        print(
            f"  Latest Offset  : {latest_offset}"
        )

        print(
            f"  Description    : {description[:100]}"
        )


    # --------------------------------------------------------
    # Total input rows
    # --------------------------------------------------------

    print(
        "\n📥 Total input rows read from source: "
        f"{total_input_rows}"
    )


# ============================================================
# Step 10: Check rescued / malformed records
# ============================================================

print("\n" + "=" * 70)
print("RESCUED DATA CHECK")
print("=" * 70)


table_columns = (
    spark.table(target_table)
    .columns
)


if "_rescued_data" in table_columns:

    rescued_count = (
        spark.table(target_table)
        .filter(
            F.col("_rescued_data").isNotNull()
        )
        .count()
    )

    if rescued_count > 0:

        print(
            f"⚠️ {rescued_count} row(s) "
            "contain rescued/malformed data."
        )

        print(
            "Check the '_rescued_data' column."
        )

    else:

        print(
            "✅ No rescued/malformed records found."
        )

else:

    print(
        "ℹ️ '_rescued_data' column does not exist."
    )


# ============================================================
# Step 11: Display latest records
# ============================================================

print("\n" + "=" * 70)
print("LATEST BRONZE RECORDS")
print("=" * 70)

(
    spark.table(target_table)
    .orderBy(
        F.col("_ingested_at").desc()
    )
    .show(
        10,
        truncate=False
    )
)


# ============================================================
# Step 12: Final status
# ============================================================

print("\n" + "=" * 70)
print("LOAD SUMMARY")
print("=" * 70)

print(
    f"Source Path      : {source_path}"
)

print(
    f"Target Table     : {target_table}"
)

print(
    f"Target Path      : {target_path}"
)

print(
    f"Rows Before      : {count_before}"
)

print(
    f"Rows After       : {count_after}"
)

print(
    f"Rows Inserted    : {rows_inserted}"
)

print(
    f"Input Rows Read  : {total_input_rows if progress_list else 0}"
)

print(
    f"Checkpoint       : {checkpoint_path}"
)

print(
    "\n✅ Incremental Bronze load completed successfully!"
)